In [0]:
#************************************************************************************************************************************
#*                                                                                                                                  *
#*   NOTEBOOK:     Column_Validation.                                                                                          *
#*                                                                                                                                  *
#*   DESCRIPTION:                                                                                                                   *
#*                                                                                                                                  *
#*                                                                                                                                  *
#*   INPUT PARMS:                                                                                                                   *
#*                                                                                                                                  *
#*                                                                                                                                  *
#*   INPUT FILES:                                                                                                                   *
#*                                                                                                                                  *
#*                                                                                                                                  *
#*   OUTPUT FILE:                                                                                                                   *
#*                                                                                                                                  *
#*   EXITS:       0 - success                                                                                                       *
#*                <> 0 - failure                                                                                                    *
#*                                                                                                                                  *
#************************************************************************************************************************************
#*                                                                                                                                  *
#*                                                 Modification Log                                                                 *
#*                                                                                                                                  *
#*    Date     CO                 Author              Description                                                                   *
#* ---------- ------------------  -----------------   ------------------------------------------------------------------------------*
#*10/22/2025 000000000000000001  Ahmad Afzaal        Implement logic to validate ENRL_SPAN_TYP field to expected values             *
#                                                     EDW_VEN116FA_PartA_Staging we are expecting "A" EDW_VEN116FA_PartD01_Staging  *
#                                                     we are expecting "D"10/22/2025.- Implement logic to validate ENRL_SPAN_TYP    *
#                                                     field to expected values EDW_VEN116FA_PartA_Staging we are expecting "A"      *
#                                                     EDW_VEN116FA_PartD01_Staging we are expecting "D"                             *
#************************************************************************************************************************************

In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("base_path_col", "s3://gia-stg-oh-ue1-data-raw/haven/inbound/VE_EDW/process/ETL_Elig", "Base Path col")

dbutils.widgets.text("column_name", "ENRL_SPAN_TYP", "Column to Validate")
dbutils.widgets.text("error_report_table", "oh_apm_stg.vendor_extracts.error_load_report_log_cpc_Stg_Elig", "Error Report Table")


# Widgets for file prefixes
dbutils.widgets.text("file1_prefix", "ODM.EDW.VEN116FA.MONTHLY.ASCII.PROD.PART.A", "File 1 Prefix")
dbutils.widgets.text("file2_prefix", "ODM.EDW.VEN116FA.MONTHLY.ASCII.PROD.PART.D01", "File 2 Prefix")
dbutils.widgets.text("file3_prefix", "ODM.EDW.VEN116FA.MONTHLY.ASCII.PROD.PART.E", "File 3 Prefix")
dbutils.widgets.text("file4_prefix", "ODM.EDW.VEN116FA.MONTHLY.ASCII.PROD.PART.N", "File 4 Prefix")
dbutils.widgets.text("file5_prefix", "ODM.EDW.VEN116FA.MONTHLY.ASCII.PROD.PART.I", "File 5 Prefix")
dbutils.widgets.text("file6_prefix", "ODM.EDW.VEN116FA.MONTHLY.ASCII.PROD.PART.K", "File 6 Prefix")
dbutils.widgets.text("file7_prefix", "ODM.EDW.VEN116FA.MONTHLY.ASCII.PROD.PART.L", "File 7 Prefix")
dbutils.widgets.text("file8_prefix", "ODM.EDW.VEN116FA.MONTHLY.ASCII.PROD.PART.O", "File 8 Prefix")

FILE_PREFIXES = [
    FILE1_PREFIX, FILE2_PREFIX, FILE3_PREFIX, FILE4_PREFIX,
    FILE5_PREFIX, FILE6_PREFIX, FILE7_PREFIX, FILE8_PREFIX
]



In [0]:

# Read widget values
BASE = dbutils.widgets.get("base_path_col").rstrip("/")
COL_NAME = dbutils.widgets.get("column_name")
ERROR_TABLE = dbutils.widgets.get("error_report_table")

FILE1_PREFIX = dbutils.widgets.get("file1_prefix")
FILE2_PREFIX = dbutils.widgets.get("file2_prefix")
FILE3_PREFIX = dbutils.widgets.get("file3_prefix")
FILE4_PREFIX = dbutils.widgets.get("file4_prefix")
FILE5_PREFIX = dbutils.widgets.get("file5_prefix")
FILE6_PREFIX = dbutils.widgets.get("file6_prefix")
FILE7_PREFIX = dbutils.widgets.get("file7_prefix")
FILE8_PREFIX = dbutils.widgets.get("file8_prefix")


print(f"Base Path: {BASE}")
print(f"Column: {COL_NAME}")
print(f"Error Table: {ERROR_TABLE}")
print(f"File Prefixes: {FILE_PREFIXES}")


In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from datetime import datetime
import re

nb_start = datetime.now()
nb_start_str = nb_start.strftime("%Y-%m-%d %H:%M:%S")

# Allowed values for each file
ALLOWED_V1 = ['A']
ALLOWED_V2 = ['D']
ALLOWED_V3 = ['E']
ALLOWED_V4 = ['N']
ALLOWED_V5 = ['I']
ALLOWED_V6 = ['K']
ALLOWED_V7 = ['L']
ALLOWED_V8 = ['O']

PREVALIDATION_TASK_KEY = "PreValidation"  # keep this single source of truth

# ----------------------------
# Helpers
# ----------------------------
def try_get_date_received_from_prevalidation():
    """Try to retrieve the date from a previous task value."""
    try:
        val = dbutils.jobs.taskValues.get(
            taskKey=PREVALIDATION_TASK_KEY,  # use the constant everywhere
            key="ctl_received_ts",
            debugValue=None
        )
        return str(val).strip() if val is not None and str(val).strip() else None
    except Exception:
        return None

def fallback_date_received(*dates):
    d8 = [d for d in dates if d and len(d) == 8 and d.isdigit()]
    if d8:
        best = max(d8)
        return f"{best[0:4]}-{best[4:6]}-{best[6:8]} 00:00:00"
    return nb_start_str

def read_csv_gz(path: str):
    # Read only the column to validate; avoids schema inference for unused columns
    return (
        spark.read
             .option("header", "true")
             .option("delimiter", "|")
             .csv(path)
             .select(COL_NAME)  # project only the needed column
    )

def resolve_files_by_prefix(base_dir: str, prefixes: list, ext: str = ".gz"):
    results = {}
    try:
        entries = dbutils.fs.ls(base_dir)
    except Exception as e:
        err = f"Unable to list dir {base_dir}: {e}"
        for p in prefixes:
            results[p] = (None, None, None, err)
        return results

    # Build per-prefix regex up front
    regex_by_prefix = {
        p: re.compile(rf"^({re.escape(p)})\.(\d{{6}}|\d{{8}}){re.escape(ext)}$")
        for p in prefixes
    }

    # Collect candidates for each prefix
    buckets = {p: [] for p in prefixes}
    for it in entries:
        name = it.name
        for p, rx in regex_by_prefix.items():
            m = rx.match(name)
            if m:
                date_digits = m.group(2)
                buckets[p].append((it.path, name, int(date_digits), date_digits, it.modificationTime))

    # Select the best candidate for each prefix
    for p in prefixes:
        cands = buckets[p]
        if not cands:
            results[p] = (None, None, None, f"No file for prefix '{p}' with 6/8-digit date and {ext}")
        else:
            cands.sort(key=lambda x: (x[2], x[4]), reverse=True)
            best = cands[0]
            # (path, name, date_str, err)
            results[p] = (best[0], best[1], best[3], None)

    return results

def summarize_invalids_single_row(df, file_name: str, target_col: str, allowed_values: list,
                                  date_received: str, start_str: str, end_str: str):
    # Locate the actual column name case-insensitively
    col_actual = next((c for c in df.columns if c.lower() == target_col.lower()), None)

    # If target column missing, all rows considered invalid
    if col_actual is None:
        total = df.count()
        return {
            "File_Name": file_name,
            "Date_Received": date_received,
            "Start_Load_Date": start_str,
            "End_Load_Date": end_str,
            "Row_Number": int(total),
            "Error_Description": f"Column '{target_col}' not found; all {total} rows considered invalid."
        }

    allowed_upper = [v.upper() for v in allowed_values]

    # Project, normalize case once; cache to reuse for total/invalid and breakdown
    df_norm = df.select(F.upper(F.col(col_actual)).alias("val"))
    df_norm_cached = df_norm.cache()

    # Compute total and invalid_total in one aggregation on the cached projection
    agg = (df_norm_cached
           .agg(
               F.count("*").alias("total"),
               F.sum((~F.col("val").isin(allowed_upper)).cast("int")).alias("invalid_total")
           )
           .first())

    total = int(agg["total"])
    invalid_total = int(agg["invalid_total"]) if agg["invalid_total"] is not None else 0

    if invalid_total == 0:
        df_norm_cached.unpersist()
        return None

    # Compute counts of each invalid value (on the cached projection)
    counts_df = (
        df_norm_cached
        .where(~F.col("val").isin(allowed_upper))
        .groupBy(F.coalesce(F.col("val"), F.lit("&lt;NULL&gt;")).alias("value"))
        .agg(F.count(F.lit(1)).alias("cnt"))
        .orderBy(F.desc("cnt"), F.asc("value"))
    )

    parts = [f"Invalid Value = '{r['value']}' (count={int(r['cnt'])})" for r in counts_df.collect()]
    df_norm_cached.unpersist()
    description = "; ".join(parts)

    return {
        "File_Name": file_name,
        "Date_Received": date_received,
        "Start_Load_Date": start_str,
        "End_Load_Date": end_str,
        "Row_Number": invalid_total,
        "Error_Description": description
    }

# ----------------------------
# Resolve files (single ls)
# ----------------------------
prefixes = [FILE1_PREFIX, FILE2_PREFIX, FILE3_PREFIX, FILE4_PREFIX,
            FILE5_PREFIX, FILE6_PREFIX, FILE7_PREFIX, FILE8_PREFIX]

allowed_by_prefix = {
    FILE1_PREFIX: ALLOWED_V1,
    FILE2_PREFIX: ALLOWED_V2,
    FILE3_PREFIX: ALLOWED_V3,
    FILE4_PREFIX: ALLOWED_V4,
    FILE5_PREFIX: ALLOWED_V5,
    FILE6_PREFIX: ALLOWED_V6,
    FILE7_PREFIX: ALLOWED_V7,
    FILE8_PREFIX: ALLOWED_V8,
}

resolved = resolve_files_by_prefix(BASE, prefixes, ".gz")

# Log which files were found
for i, p in enumerate(prefixes, start=1):
    fpath, fname, fdate, ferr = resolved[p]
    print(f"File{i} -> {fname or ferr}")

# ----------------------------
# Determine Date_Received (same logic)
# ----------------------------
DATE_RECEIVED = try_get_date_received_from_prevalidation()
if not DATE_RECEIVED:
    DATE_RECEIVED = fallback_date_received(*(resolved[p][2] for p in prefixes))
print(f"Date_Received used for logging: {DATE_RECEIVED}")

rows_to_write = []

def validate_file(file_path, file_name, prefix, allowed_values):
    nb_end_str_local = datetime.now().strftime("%Y-%m-%d %H:%M:%S")  # maintain semantics, per validation block

    if file_path is None:
        rows_to_write.append({
            "File_Name": f"{prefix}.&lt;date&gt;.gz",
            "Date_Received": DATE_RECEIVED,
            "Start_Load_Date": nb_start_str,
            "End_Load_Date": nb_end_str_local,
            "Row_Number": 0,
            "Error_Description": "File not found by prefix."
        })
        return

    try:
        df = read_csv_gz(file_path)
        r = summarize_invalids_single_row(
            df=df,
            file_name=file_name,
            target_col=COL_NAME,
            allowed_values=allowed_values,
            date_received=DATE_RECEIVED,
            start_str=nb_start_str,
            end_str=nb_end_str_local
        )
        if r:
            rows_to_write.append(r)
    except Exception as e:
        rows_to_write.append({
            "File_Name": file_name or f"{prefix}.&lt;date&gt;.gz",
            "Date_Received": DATE_RECEIVED,
            "Start_Load_Date": nb_start_str,
            "End_Load_Date": nb_end_str_local,
            "Row_Number": 0,
            "Error_Description": f"Read error: {str(e)}"
        })

# Run validations for all files
for p in prefixes:
    fpath, fname, fdate, ferr = resolved[p]
    validate_file(fpath, fname, p, allowed_by_prefix[p])

# ----------------------------
# Report
# ----------------------------
if rows_to_write:
    display(spark.createDataFrame(
        rows_to_write,
        schema=T.StructType([
            T.StructField("File_Name", T.StringType()),
            T.StructField("Date_Received", T.StringType()),
            T.StructField("Start_Load_Date", T.StringType()),
            T.StructField("End_Load_Date", T.StringType()),
            T.StructField("Row_Number", T.LongType()),
            T.StructField("Error_Description", T.StringType()),
        ])
    ))
else:
    print("✅ No errors found in the validation checks.")

UPLOAD_TO_TABLE = len(rows_to_write) > 0
ROWS_PREPARED = len(rows_to_write)

dbutils.jobs.taskValues.set(key="error_load_report_flag", value=UPLOAD_TO_TABLE)
print(f"Prepared {ROWS_PREPARED} row(s). UPLOAD_TO_TABLE = {UPLOAD_TO_TABLE}")


In [0]:
from pyspark.sql.types import StructType, StructField, StringType, LongType

Col_Validation_error = False

if UPLOAD_TO_TABLE:
    error_schema = StructType([
        StructField("File_Name", StringType(), True),
        StructField("Date_Received", StringType(), True),
        StructField("Start_Load_Date", StringType(), True),
        StructField("End_Load_Date", StringType(), True),
        StructField("Row_Number", LongType(), True),
        StructField("Error_Description", StringType(), True),
    ])

    error_df = spark.createDataFrame(rows_to_write, schema=error_schema)

    if error_df.count() > 0:
        Col_Validation_error = True
        error_df.write.format("delta").mode("append").saveAsTable(ERROR_TABLE)
        print(f"❌ Validation errors found. Wrote {error_df.count()} row(s) to {ERROR_TABLE}")
    else:
        print("✅ No validation errors found. Skipping upload.")
else:
    print("⚠️ UPLOAD_TO_TABLE is False. Skipping error upload.")


print(f"Col_Validation_error = {Col_Validation_error}")
dbutils.jobs.taskValues.set(key="Col_Validation_error", value=Col_Validation_error)
